# M56c grouped FP16-storage sensitivity

This is a **no-training, no-full-evaluation** diagnostic. It rounds one MonoDGP architectural group at a time, then all groups except one, and measures raw-output drift on fixed Chen validation image `000001`. Run all cells on a GPU and return the final JSON and CSV.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from collections import deque
import json, os, shlex, shutil, subprocess, sys
MOBILE_REPO=Path('/content/mobile_adas3d')
MONODGP_REPO=Path('/content/MonoDGP_M56C')
MONODGP_COMMIT='aa059a18214aebf644510e7f0793971b403f9d14'
DRIVE_DATASET_ROOT=Path('/content/drive/MyDrive/datasets/kitti')
LOCAL_DATASET_ROOT=Path('/content/kitti')
SPLIT_DIR=Path('/content/drive/MyDrive/mobile_adas3d_splits/kitti_chen')
DATASET_ROOT=Path('/content/monodgp_kitti_m56c')
M54_ROOT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/challengers/monodgp_m54')
M54_SELECTION=M54_ROOT/'product_checkpoint_sweep/m54_product_selection.json'
M55_ROOT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/compression/monodgp_m55_feasibility')
M55_GATE=M55_ROOT/'m55_feasibility_gate.json'
M55_PROFILE=M55_ROOT/'m55_native_baseline_profile.json'
M55_AUDIT=M55_ROOT/'m55_operator_export_audit.json'
M56B_ROOT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/compression/monodgp_m56b_selective_fp16_storage')
M56B_MANIFEST=M56B_ROOT/'m56b_compression_manifest.json'
M56B_SMOKE=M56B_ROOT/'m56b_selective_fp16_storage_smoke.json'
OUTPUT_ROOT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/compression/monodgp_m56c_group_sensitivity')
LOG_DIR=OUTPUT_ROOT/'colab_logs'
def run(command,cwd=None,env=None):
    command=[str(x) for x in command]; print('+',shlex.join(command),flush=True)
    merged=os.environ.copy(); merged.update(env or {})
    result=subprocess.run(command,cwd=cwd,env=merged)
    if result.returncode: raise RuntimeError(f'Exit {result.returncode}: {shlex.join(command)}')
def run_logged(command,cwd,log_path,env=None):
    command=[str(x) for x in command]; print('+',shlex.join(command),flush=True)
    log_path=Path(log_path); log_path.parent.mkdir(parents=True,exist_ok=True)
    merged=os.environ.copy(); merged.update(env or {}); tail=deque(maxlen=120)
    with log_path.open('w',encoding='utf-8') as log:
        process=subprocess.Popen(command,cwd=cwd,env=merged,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            print(line,end='',flush=True); log.write(line); tail.append(line.rstrip())
        code=process.wait()
    if code: raise RuntimeError(f'Exit {code}; full log={log_path}\n'+'\n'.join(tail))
    return log_path
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
run(['nvidia-smi'])


In [ ]:
# Fetch exact sources, apply audited compatibility patches, and build deformable attention.
if not MOBILE_REPO.exists():
    run(['git','clone','https://github.com/Ali-RT/mobile_adas3d.git',MOBILE_REPO])
else:
    run(['git','pull','--ff-only'],cwd=MOBILE_REPO)
if not MONODGP_REPO.exists():
    run(['git','clone','https://github.com/PuFanqi23/MonoDGP.git',MONODGP_REPO])
run(['git','fetch','--all'],cwd=MONODGP_REPO)
run(['git','checkout',MONODGP_COMMIT],cwd=MONODGP_REPO)
run([sys.executable,'-m','pip','install','-q','pyyaml','scipy','opencv-python-headless','numba','scikit-image','scikit-learn','tqdm','ninja','pandas'])
run([sys.executable,'scripts/patch_monodgp_colab_compat.py','--monodgp-repo',MONODGP_REPO],cwd=MOBILE_REPO)
run([sys.executable,'scripts/patch_monodgp_m54_training.py','--monodgp-repo',MONODGP_REPO],cwd=MOBILE_REPO)
changed=set(subprocess.run(['git','diff','--name-only'],cwd=MONODGP_REPO,check=True,capture_output=True,text=True).stdout.splitlines())
expected={
    'lib/datasets/kitti/kitti_dataset.py',
    'lib/helpers/save_helper.py',
    'lib/helpers/trainer_helper.py',
    'lib/models/monodgp/ops/modules/ms_deform_attn.py',
    'lib/models/monodgp/ops/setup.py',
    'lib/models/monodgp/ops/src/cuda/ms_deform_attn_cuda.cu',
    'tools/train_val.py',
}
if changed != expected: raise RuntimeError(f'Unexpected patched source set: {changed}')
ops=MONODGP_REPO/'lib/models/monodgp/ops'
shutil.rmtree(ops/'build',ignore_errors=True)
run([sys.executable,'setup.py','build','install'],cwd=ops,env={'MAX_JOBS':'2'})
run([sys.executable,'-c','import torch, MultiScaleDeformableAttention; print(torch.__version__,torch.version.cuda,torch.cuda.get_device_name(0))'],cwd=MONODGP_REPO)


In [ ]:
# Create the canonical Chen-split KITTI view without copying images.
def resolve(root,names):
    for name in names:
        path=root/name
        if path.is_dir(): return path
sources={
    key:resolve(LOCAL_DATASET_ROOT,names) or resolve(DRIVE_DATASET_ROOT,names)
    for key,names in {
        'image_2':['training/image_2','training/image_02'],
        'label_2':['training/label_2','training/label_02'],
        'calib':['training/calib'],
    }.items()
}
if any(path is None for path in sources.values()): raise FileNotFoundError(sources)
(DATASET_ROOT/'training').mkdir(parents=True,exist_ok=True)
(DATASET_ROOT/'ImageSets').mkdir(parents=True,exist_ok=True)
for name,target in sources.items():
    link=DATASET_ROOT/'training'/name
    if link.is_symlink() and link.resolve()==target.resolve(): continue
    if link.exists() or link.is_symlink(): raise RuntimeError(f'Refusing to replace {link}')
    link.symlink_to(target,target_is_directory=True)
for split in ('train','val'):
    shutil.copy2(SPLIT_DIR/f'{split}.txt',DATASET_ROOT/'ImageSets'/f'{split}.txt')
assert len((DATASET_ROOT/'ImageSets/train.txt').read_text().splitlines())==3712
assert len((DATASET_ROOT/'ImageSets/val.txt').read_text().splitlines())==3769
for required in (M54_SELECTION,M55_GATE,M55_PROFILE,M55_AUDIT,M56B_MANIFEST,M56B_SMOKE):
    if not required.is_file(): raise FileNotFoundError(required)


In [ ]:
# Freeze provenance and generate the alias-consistent group matrix.
PREPARE_LOG=run_logged([
    sys.executable,'-u','scripts/prepare_monodgp_m56c_group_sensitivity.py',
    '--monodgp-repo',MONODGP_REPO,
    '--dataset-root',DATASET_ROOT,
    '--m54-selection',M54_SELECTION,
    '--m55-gate',M55_GATE,
    '--m55-profile',M55_PROFILE,
    '--m55-audit',M55_AUDIT,
    '--m56b-manifest',M56B_MANIFEST,
    '--m56b-smoke',M56B_SMOKE,
    '--output-root',OUTPUT_ROOT,
],MOBILE_REPO,LOG_DIR/'m56c_prepare.log')
MANIFEST=OUTPUT_ROOT/'m56c_group_sensitivity_manifest.json'
manifest=json.loads(MANIFEST.read_text())
assert manifest['diagnostic_authorized'] and all(manifest['preparation_gate_results'].values())
assert manifest['full_evaluation_authorized'] is False
assert manifest['offline_compression_candidate_selected'] is False
print('Diagnostic candidates:',manifest['candidate_count'])
import pandas as pd
display(pd.DataFrame([{'group':name,**row} for name,row in manifest['group_inventory']['groups'].items()]))


In [ ]:
# Run every singleton and complement policy on fixed image 000001.
REPORT=OUTPUT_ROOT/'m56c_group_sensitivity.json'
DIAGNOSTIC_LOG=run_logged([
    sys.executable,'-u','scripts/diagnose_monodgp_m56c_fp16_groups.py',
    '--monodgp-repo',MONODGP_REPO,
    '--manifest',MANIFEST,
    '--output',REPORT,
],MOBILE_REPO,LOG_DIR/'m56c_group_sensitivity.log')
report=json.loads(REPORT.read_text())
assert report['all_diagnostic_gates_passed']
assert report['full_evaluation_authorized'] is False
print(json.dumps(report['analysis'],indent=2))
RESULTS_CSV=REPORT.with_suffix('.csv')
table=pd.read_csv(RESULTS_CSV)
display(table[['candidate','mode','fp32_held_group','projected_parameter_size_ratio','all_parity_limits_passed','failed_output_families','pred_depth_max_abs']])
print('Return:',MANIFEST,REPORT,RESULTS_CSV)


## Final stop — return M56c evidence

Return `m56c_group_sensitivity_manifest.json`, `m56c_group_sensitivity.json`, and `m56c_group_sensitivity.csv`. This notebook contains no complete-validation cell and selects no compressed model. Do not run a 3,769-image evaluation from M56c.
